# PrintFarmEnv — GRPO Dispatcher Training

**Stack:** Unsloth (fast LoRA) + TRL `GRPOTrainer` + PrintFarmEnv verifier

**Flow:**
```
SFT checkpoint  →  collect env rollouts  →  GRPO optimises against P&L reward  →  eval
```

**Organiser framing (Help Guide §10–11):**
> "If the task is verifiable, build the verifier first, then plug that verifier into RL training."
> The intended stack is TRL (GRPOTrainer) + Unsloth + OpenEnv.

Our env's `step_reward_usd` (dollar P&L delta per dispatcher decision) is the verifier reward.

---
**Swap model:** change `MODEL_ID` cell. Everything else adapts automatically via `config/models.yaml`.

## 0 · Install dependencies

In [ ]:
# ::: Code Generated by Copilot c5e8a34d-4b9b-41a0-b82e-d3e7c2f46901. This comment will be removed automatically after the file is saved :::
# ── Install once per Colab session ──────────────────────────────────────────────────────
import subprocess, sys

def run(cmd):
    subprocess.run(cmd, shell=True, check=True)

run('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
run('pip install -q "trl>=0.8.0" "datasets>=2.18" "peft>=0.10" "accelerate>=0.28"')
run('pip install -q "openenv-core>=0.1" pydantic fastapi uvicorn')

# Clone repo if running in Colab (skip locally)
import os
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    if not os.path.exists('PrintFarmEnv'):
        run('git clone https://github.com/kshitizP/PrintFarmEnv.git')
    os.chdir('PrintFarmEnv')
    sys.path.insert(0, '.')

print('Dependencies ready.')

## 1 · Configuration  ← one-cell model swap

In [ ]:
# ── Model swap: change MODEL_ID and nothing else needs to change ─────────────
MODEL_ID   = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # fast smoke test
# MODEL_ID = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"  # primary
# MODEL_ID = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"  # secondary
# MODEL_ID = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"    # tertiary

SFT_CHECKPOINT = None   # set to "./sft_output" to start from SFT checkpoint

# ── GRPO training hyperparams ────────────────────────────────────────────────
TASKS           = ["task_1"]                # start narrow; add task_2/3 after smoke test
EPISODES        = 20                        # reference policy episodes per task (offline rollouts)
GROUP_SIZE      = 8                         # GRPO group size (G completions per prompt)
MAX_STEPS       = 50                        # GRPO update steps (increase to 200+ after loop stable)
BATCH_SIZE      = 4                         # per-device batch
GRAD_ACCUM      = 2
LR              = 5e-6
MAX_NEW_TOKENS  = 128
LORA_RANK       = 16
SEED            = 42

OUTPUT_DIR      = "./grpo_output"
ROLLOUT_DIR     = "./data/grpo"

print(f"Model : {MODEL_ID}")
print(f"Tasks : {TASKS}")
print(f"Steps : {MAX_STEPS}  |  Group : {GROUP_SIZE}  |  LR : {LR}")

## 2 · Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

base_path = SFT_CHECKPOINT if SFT_CHECKPOINT else MODEL_ID

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = base_path,
    max_seq_length = 2048,
    dtype          = None,          # auto-detect
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_RANK * 2,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = SEED,
)

print(f"Loaded: {base_path}")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 3 · Collect reference rollouts (offline prompt dataset)

In [ ]:
import os, sys
sys.path.insert(0, '.')

from scripts.grpo_rollout import collect_rollouts, save_dataset, load_as_hf_dataset
from scripts.grpo_rollout import _clairvoyant_policy_fn

# Collect reference trajectories using clairvoyant policy
# These give us prompts (env observations) + ground-truth actions + step rewards
print("Collecting reference rollouts (clairvoyant policy)...")
records = collect_rollouts(
    policy_fn  = _clairvoyant_policy_fn,
    tasks      = TASKS,
    n_episodes = EPISODES,
    base_seed  = SEED,
    verbose    = False,
)
save_dataset(records, ROLLOUT_DIR)
print(f"Collected {len(records)} step records across {EPISODES} episodes.")

## 4 · Build HuggingFace Dataset

In [ ]:
from datasets import Dataset

raw_ds = load_as_hf_dataset(f"{ROLLOUT_DIR}/grpo_steps.jsonl")

# Format prompts using the model's chat template
def format_prompt(example):
    messages = example["prompt_messages"]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize        = False,
        add_generation_prompt = True,
    )
    return {"prompt": prompt}

dataset = raw_ds.map(format_prompt)

# Keep columns GRPOTrainer needs + extra columns for the reward fn
keep = ["prompt", "completion", "reward", "task_id", "seed", "step_idx", "prior_actions"]
dataset = dataset.select_columns(keep)

print(dataset)
print("Sample prompt (truncated):")
print(dataset[0]["prompt"][:400], "...")
print(f"\nReward range: {min(dataset['reward']):.3f} – {max(dataset['reward']):.3f}")
print(f"Mean step reward: {sum(dataset['reward'])/len(dataset):.4f}")

## 5 · Define reward function

Two reward functions run in parallel — reward hacking mitigation (Help Guide §8, FAQ #13):
- **`env_reward`** — env's dollar P&L delta (the verifier)
- **`format_reward`** — parses completion as valid JSON FarmAction (+0.2 / -0.2)

Multiple independent reward functions reduce the chance of the model exploiting a single signal.

In [ ]:
import json, re
from scripts.grpo_rollout import make_env_reward_fn, parse_action
from printfarm_env.models import FarmActionEnum

# ── Reward 1: env P&L verifier ───────────────────────────────────────────────
env_reward_fn = make_env_reward_fn()

# ── Reward 2: format compliance (+0.2 valid JSON FarmAction, -0.2 otherwise) ─
VALID_ACTIONS = {a.value for a in FarmActionEnum}

def format_reward_fn(prompts, completions, **kwargs):
    rewards = []
    for c in completions:
        try:
            data = json.loads(c.strip())
            if data.get("action") in VALID_ACTIONS:
                rewards.append(0.2)
            else:
                rewards.append(-0.2)
        except Exception:
            rewards.append(-0.2)
    return rewards

# ── Reward 3: penalise pure WAIT spam (reward hacking guard) ─────────────────
def anti_wait_spam_fn(prompts, completions, **kwargs):
    """Small penalty if completion is WAIT when env has actionable state.
    Prevents model from learning 'always WAIT' as a local optimum.
    """
    rewards = []
    for c in completions:
        try:
            data = json.loads(c.strip())
            rewards.append(-0.1 if data.get("action") == "WAIT" else 0.0)
        except Exception:
            rewards.append(0.0)
    return rewards

print("Reward functions defined:")
print("  1. env_reward_fn      — step_reward_usd from PrintFarmEnv verifier")
print("  2. format_reward_fn   — valid JSON FarmAction check (+0.2 / -0.2)")
print("  3. anti_wait_spam_fn  — WAIT-spam penalty (-0.1 per WAIT action)")

## 6 · Configure and run GRPOTrainer

In [ ]:
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    # ── Core ─────────────────────────────────────────────────────────────────
    output_dir                    = OUTPUT_DIR,
    max_steps                     = MAX_STEPS,
    per_device_train_batch_size   = BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM,
    learning_rate                 = LR,
    # ── GRPO-specific ────────────────────────────────────────────────────────
    num_generations               = GROUP_SIZE,   # G completions per prompt
    max_new_tokens                = MAX_NEW_TOKENS,
    temperature                   = 0.9,          # diversity for group sampling
    # ── Efficiency (Unsloth-friendly) ────────────────────────────────────────
    bf16                          = True,
    optim                         = "adamw_8bit",
    lr_scheduler_type             = "cosine",
    warmup_ratio                  = 0.1,
    seed                          = SEED,
    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps                 = 5,
    save_steps                    = 25,
    report_to                     = "none",       # swap to "wandb" for full tracking
)

trainer = GRPOTrainer(
    model         = model,
    tokenizer     = tokenizer,
    reward_funcs  = [env_reward_fn, format_reward_fn, anti_wait_spam_fn],
    args          = grpo_config,
    train_dataset = dataset,
)

print("GRPOTrainer ready.")
print(f"  Dataset rows  : {len(dataset)}")
print(f"  Max steps     : {MAX_STEPS}")
print(f"  Group size    : {GROUP_SIZE}")
print(f"  Reward fns    : env P&L + format compliance + anti-WAIT-spam")

In [ ]:
# ── TRAIN ────────────────────────────────────────────────────────────────────
# Monitor: watch reward/env_reward go up, reward/format_reward stay high,
# and reward/anti_wait_spam stay near 0 (model not spamming WAIT).
# Help Guide §15: watch individual reward columns, not just average reward.
trainer.train()
print("Training complete.")

## 7 · Inspect generations during/after training

Help Guide §15: *"Also inspect actual generations during training. A rising reward is not enough if the model is learning to exploit bugs."*

In [ ]:
import json
from scripts.grpo_rollout import obs_to_prompt_messages, parse_action
from printfarm_env.env import PrintFarmEnvironment

FastLanguageModel.for_inference(model)  # enable faster inference

env  = PrintFarmEnvironment()
obs  = env.reset(seed=SEED, task_id=TASKS[0])

print(f"Sampling 5 model actions on {TASKS[0]} (step 0):")
print("-" * 60)

messages = obs_to_prompt_messages(obs)
prompt   = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

for i in range(5):
    out = model.generate(
        **inputs,
        max_new_tokens = MAX_NEW_TOKENS,
        temperature    = 0.9,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )
    decoded   = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    action    = parse_action(decoded)
    print(f"  [{i+1}] raw: {decoded[:80]!r}")
    print(f"       action: {action.model_dump_json()}")
    print()

## 8 · Evaluate trained model vs baselines

In [ ]:
# ::: Code Generated by Copilot c5e8a34d-4b9b-41a0-b82e-d3e7c2f46901. This comment will be removed automatically after the file is saved :::
from printfarm_env.env import PrintFarmEnvironment
from scripts.grpo_rollout import parse_action, obs_to_prompt_messages
from baselines.naive_greedy import naive_action
from baselines.clairvoyant_greedy import clairvoyant_action

EVAL_EPISODES = 10
EVAL_TASKS    = TASKS

def run_model_eval(task_id, n_episodes, seed=SEED):
    env = PrintFarmEnvironment()
    scores = []
    for ep in range(n_episodes):
        obs = env.reset(seed=seed + ep, task_id=task_id)
        while not obs.done:
            messages = obs_to_prompt_messages(obs)
            prompt   = tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            out    = model.generate(
                **inputs, max_new_tokens=MAX_NEW_TOKENS,
                temperature=0.2, do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
            decoded = tokenizer.decode(
                out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
            )
            obs = env.step(parse_action(decoded))
        scores.append(obs.reward)
    return scores

def run_naive_eval(task_id, n_episodes, seed=SEED):
    env = PrintFarmEnvironment()
    scores = []
    for ep in range(n_episodes):
        obs = env.reset(seed=seed + ep, task_id=task_id)
        while not obs.done:
            obs = env.step(naive_action(obs))
        scores.append(obs.reward)
    return scores

def run_clairvoyant_eval(task_id, n_episodes, seed=SEED):
    env = PrintFarmEnvironment()
    scores = []
    for ep in range(n_episodes):
        obs = env.reset(seed=seed + ep, task_id=task_id)
        while not obs.done:
            obs = env.step(clairvoyant_action(env))
        scores.append(obs.reward)
    return scores

results = {}
for task_id in EVAL_TASKS:
    print(f"\n{task_id}:")
    naive_scores  = run_naive_eval(task_id, EVAL_EPISODES)
    clair_scores  = run_clairvoyant_eval(task_id, EVAL_EPISODES)
    model_scores  = run_model_eval(task_id, EVAL_EPISODES)

    naive_mean    = sum(naive_scores)  / len(naive_scores)
    clair_mean    = sum(clair_scores)  / len(clair_scores)
    model_mean    = sum(model_scores)  / len(model_scores)
    gap_captured  = (model_mean - naive_mean) / max(clair_mean - naive_mean, 1e-6)

    results[task_id] = {
        "naive":  naive_mean, "clairvoyant": clair_mean,
        "model":  model_mean, "gap_captured_pct": gap_captured * 100,
    }
    print(f"  naive       : {naive_mean:.4f}")
    print(f"  clairvoyant : {clair_mean:.4f}")
    print(f"  GRPO model  : {model_mean:.4f}")
    print(f"  gap captured: {gap_captured*100:.1f}%")

## 9 · Plot reward curves

In [ ]:
import matplotlib.pyplot as plt
import json
from pathlib import Path

# Load trainer log history
log_history = trainer.state.log_history

steps   = [x["step"]            for x in log_history if "rewards/env_reward_fn" in x]
env_r   = [x["rewards/env_reward_fn"] for x in log_history if "rewards/env_reward_fn" in x]
fmt_r   = [x.get("rewards/format_reward_fn", 0) for x in log_history if "rewards/env_reward_fn" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(steps, env_r, label="env P&L reward", color="steelblue")
ax1.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax1.set_xlabel("GRPO step")
ax1.set_ylabel("Mean reward")
ax1.set_title("Env P&L reward over GRPO training")
ax1.legend()

ax2.plot(steps, fmt_r, label="format compliance", color="seagreen")
ax2.set_xlabel("GRPO step")
ax2.set_ylabel("Mean reward")
ax2.set_title("Format reward over GRPO training")
ax2.legend()

plt.tight_layout()
out_path = Path(OUTPUT_DIR) / "grpo_reward_curves.png"
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=150)
plt.show()
print(f"Saved → {out_path}")

# Before/after bar chart
fig2, ax = plt.subplots(figsize=(7, 4))
task_labels = list(results.keys())
x = range(len(task_labels))
w = 0.25
ax.bar([i - w for i in x], [results[t]["naive"]       for t in task_labels], w, label="Naive",       color="salmon")
ax.bar([i     for i in x], [results[t]["model"]       for t in task_labels], w, label="GRPO model",  color="steelblue")
ax.bar([i + w for i in x], [results[t]["clairvoyant"] for t in task_labels], w, label="Clairvoyant", color="seagreen")
ax.set_xticks(list(x))
ax.set_xticklabels(task_labels)
ax.set_ylabel("Grader score")
ax.set_title("Before (naive) vs GRPO model vs ceiling (clairvoyant)")
ax.legend()
plt.tight_layout()
bar_path = Path(OUTPUT_DIR) / "grpo_before_after.png"
plt.savefig(bar_path, dpi=150)
plt.show()
print(f"Saved → {bar_path}")

## 10 · Save model

⚠️ **Help Guide §16 warning:** Do NOT upcast a 4-bit model to 16-bit and naively merge LoRA.
Use Unsloth's `save_pretrained_merged` with `save_method="merged_16bit"` or keep adapters.

In [ ]:
# ::: Code Generated by Copilot c5e8a34d-4b9b-41a0-b82e-d3e7c2f46901. This comment will be removed automatically after the file is saved :::
from pathlib import Path

# ── Option A: Save LoRA adapters only (smallest, fastest) ────────────────────
adapter_path = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapters saved → {adapter_path}")

# ── Option B: Properly merged 16-bit model (for inference without PEFT) ───────
# Uncomment when you want the final submission model:
#
# merged_path = f"{OUTPUT_DIR}/merged_16bit"
# model.save_pretrained_merged(
#     merged_path, tokenizer, save_method="merged_16bit"
# )
# print(f"Merged 16-bit model saved → {merged_path}")

# ── Option C: Push to HF Hub ──────────────────────────────────────────────
# model.push_to_hub("kshitizP/printfarm-grpo", token="HF_TOKEN")
# tokenizer.push_to_hub("kshitizP/printfarm-grpo", token="HF_TOKEN")

print("\nTest a quick inference pass before considering this done:")
print("  Run cell 7 again to check the model still generates valid FarmAction JSON.")

## Next steps after smoke test passes

1. **Scale tasks**: add `task_2`, `task_3` to `TASKS`
2. **Scale model**: change `MODEL_ID` to `Qwen2.5-7B-Instruct-bnb-4bit`
3. **Increase steps**: set `MAX_STEPS = 200`
4. **Increase episodes**: set `EPISODES = 100` for richer prompt diversity
5. **Add curriculum**: start from task_0_1/0_2/0_3 warm-ups, then 1–3
6. **Run model-swap verification** (Llama-3.1-8B, Mistral-7B) — change `MODEL_ID`, rerun
7. **Copy curves** to `presentation/curves/` for the pitch deck